# 12 — TMDB Movie Enriched × SCL Sentiment Aggregate

Joins the movie metadata table (`02`) with the movie-level keyword sentiment scores (`11`).
Only movies that have at least one keyword are included (inner join).

**Inputs**
- `data/tmdb_movies_enriched.csv` — full movie metadata (title, genres, ratings, etc.)
- `data/tmdb_movie_sentiment_scores.csv` — per-movie keyword sentiment aggregate

**Output**
- `data/tmdb_movie_enriched_scl_aggregate.csv`
- `output/tmdb-movie-enriched-scl-aggregate/`

Only movies present in the sentiment scores table are kept (movies with keywords).

In [1]:
import shutil
from pathlib import Path

import pandas as pd
import plotly.express as px

MOVIES_FILE  = Path("data/tmdb_movies_enriched.csv")
SCORES_FILE  = Path("data/tmdb_movie_sentiment_scores.csv")
OUT_FILE     = Path("data/tmdb_movie_enriched_scl_aggregate.csv")
OUTPUT_DIR   = Path("output/tmdb-movie-enriched-scl-aggregate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
movies = pd.read_csv(MOVIES_FILE, low_memory=False)
scores = pd.read_csv(SCORES_FILE)

print(f"Movies   : {len(movies):,}  columns: {movies.columns.tolist()}")
print(f"Scores   : {len(scores):,}  columns: {scores.columns.tolist()}")

Movies   : 1,170,166  columns: ['tmdb_movie_id', 'original_title', 'popularity_2026_03_11', 'adult', 'video', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'backdrop_path', 'budget', 'homepage', 'imdb_id', 'original_language', 'overview', 'poster_path', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'keywords']
Scores   : 212,015  columns: ['tmdb_movie_id', 'keyword_count', 'scored_keyword_count', 'sentiment_mean', 'sentiment_weighted', 'sentiment_positive_frac', 'sentiment_negative_frac', 'sentiment_neutral_frac', 'dominant_polarity']


In [3]:
# Inner join — only movies with keyword sentiment scores
enriched = movies.merge(scores, on="tmdb_movie_id", how="inner")

dropped = len(movies) - len(enriched)
print(f"Movies with keywords (kept) : {len(enriched):,}")
print(f"Movies without keywords (dropped) : {dropped:,}")
print(f"Columns: {enriched.columns.tolist()}")

Movies with keywords (kept) : 212,163
Movies without keywords (dropped) : 958,003
Columns: ['tmdb_movie_id', 'original_title', 'popularity_2026_03_11', 'adult', 'video', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'backdrop_path', 'budget', 'homepage', 'imdb_id', 'original_language', 'overview', 'poster_path', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'keywords', 'keyword_count', 'scored_keyword_count', 'sentiment_mean', 'sentiment_weighted', 'sentiment_positive_frac', 'sentiment_negative_frac', 'sentiment_neutral_frac', 'dominant_polarity']


In [4]:
enriched[["title", "vote_average", "scored_keyword_count", "sentiment_weighted", "dominant_polarity"]].head(10)

,title,vote_average,scored_keyword_count,sentiment_weighted,dominant_polarity
0,Ariel,7.100,5,-0.173633,negative
1,Shadows in Paradise,7.199,2,-0.034981,positive
2,Four Rooms,5.784,8,-0.091829,positive
3,Judgment Night,6.533,4,-0.034099,negative
4,Sunday in August,7.600,2,0.252689,positive
5,Star Wars,8.204,17,-0.095172,positive
6,Finding Nemo,7.824,13,0.279904,positive
7,Forrest Gump,8.477,23,0.006779,positive
8,American Beauty,8.000,28,0.206745,positive
9,Citizen Kane,8.015,11,0.104893,positive


In [5]:
# Sentiment vs audience rating (movies with ≥5 scored keywords + ≥10 votes)
plot_df = enriched[
    (enriched["scored_keyword_count"] >= 5) &
    (enriched["vote_count"] >= 10)
].copy()

fig = px.scatter(
    plot_df.sample(min(len(plot_df), 8000), random_state=42),
    x="sentiment_weighted", y="vote_average",
    color="dominant_polarity", opacity=0.4,
    labels={
        "sentiment_weighted": "Keyword sentiment (weighted mean)",
        "vote_average": "Audience rating",
        "dominant_polarity": "Dominant polarity",
    },
    title=f"Keyword sentiment vs audience rating (n={len(plot_df):,}, ≥5 scored kw, ≥10 votes)"
)
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.show()

In [6]:
# Average sentiment by rating bucket
plot_df = enriched[enriched["vote_count"] >= 10].copy()
plot_df["rating_bucket"] = pd.cut(
    plot_df["vote_average"],
    bins=[0, 4, 5, 6, 7, 8, 10],
    labels=["<4", "4-5", "5-6", "6-7", "7-8", "8+"]
)

bucket_agg = (
    plot_df.groupby("rating_bucket", observed=True)["sentiment_weighted"]
    .agg(["mean", "median", "count"])
    .reset_index()
)
print(bucket_agg.to_string(index=False))

fig2 = px.box(
    plot_df,
    x="rating_bucket", y="sentiment_weighted",
    labels={"rating_bucket": "Rating bucket", "sentiment_weighted": "Keyword sentiment"},
    title="Keyword sentiment distribution by audience rating bucket"
)
fig2.add_hline(y=0, line_dash="dash", line_color="gray")
fig2.show()

rating_bucket      mean    median  count
           <4 -0.056371 -0.035158   1464
          4-5 -0.034609 -0.020000   4682
          5-6  0.018816  0.039484  13342
          6-7  0.042883  0.065721  19879
          7-8  0.061589  0.081954   8872
           8+  0.100475  0.132650    845


In [7]:
enriched.to_csv(OUT_FILE, index=False)
shutil.copy(OUT_FILE, OUTPUT_DIR / OUT_FILE.name)
print(f"Saved {len(enriched):,} rows → {OUT_FILE}")
print(f"Columns: {enriched.columns.tolist()}")

Saved 212,163 rows → data/tmdb_movie_enriched_scl_aggregate.csv
Columns: ['tmdb_movie_id', 'original_title', 'popularity_2026_03_11', 'adult', 'video', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'backdrop_path', 'budget', 'homepage', 'imdb_id', 'original_language', 'overview', 'poster_path', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'keywords', 'keyword_count', 'scored_keyword_count', 'sentiment_mean', 'sentiment_weighted', 'sentiment_positive_frac', 'sentiment_negative_frac', 'sentiment_neutral_frac', 'dominant_polarity']


In [8]:
import os
from datetime import date

if not os.environ.get("KAGGLE_TOKEN"):
    print("Skipping upload: set KAGGLE_TOKEN env var to upload.")
else:
    import kagglehub
    user  = kagglehub.whoami(verbose=False)["username"]
    notes = f"Pipeline run {date.today()}"
    slug  = "tmdb-movie-enriched-scl-aggregate"
    print(f"Uploading {slug} ...")
    kagglehub.dataset_upload(
        handle=f"{user}/{slug}",
        local_dataset_dir=str(OUTPUT_DIR),
        version_notes=notes,
    )
    print(f"  → kaggle.com/datasets/{user}/{slug}")

Skipping upload: set KAGGLE_TOKEN env var to upload.
